In [2]:
import sys
!{sys.executable} -m pip install llama-index

  Using cached llama_index-0.14.23-py3-none-any.whl.metadata (14 kB)
  Using cached llama_index_core-0.14.23-py3-none-any.whl.metadata (2.6 kB)
  Using cached llama_index_embeddings_openai-0.6.0-py3-none-any.whl.metadata (401 bytes)
  Using cached llama_index_llms_openai-0.7.10-py3-none-any.whl.metadata (3.0 kB)
  Using cached nltk-3.10.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached aiohttp-3.14.3-cp314-cp314-win_amd64.whl.metadata (8.5 kB)
  Using cached aiosqlite-0.22.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached banks-2.5.0-py3-none-any.whl.metadata (12 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached deprecated-1.3.1-py2.py3-none-any.whl.metadata (5.9 kB)
  Using cached dirtyjson-1.0.8-py3-none-any.whl.metadata (11 kB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached ll


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import llama_index.core
print("LlamaIndex installed successfully! Version:", llama_index.core.__version__)

LlamaIndex installed successfully! Version: 0.14.23


In [5]:
from llama_index.core import SimpleDirectoryReader

# We use "../" to go up one folder from "notebooks", then into "data/raw"
print("Loading documents...")
documents = SimpleDirectoryReader("../data/raw").load_data()

print(f"Success! Loaded {len(documents)} document(s).")

# Let's peek at what it read
print("\nHere is a preview of what the computer sees:")
print(documents[0].text)

Loading documents...
Success! Loaded 1 document(s).

Here is a preview of what the computer sees:
DeepFix is an autonomous AI framework designed to fix failing RAG systems.
It uses an AI agent named Hermes.
Agrim Sharma is the lead engineer responsible for building the retrieval pipeline.
DeepFix is a joint project between the AISC at HPI and Mittelstand Zukunftslab.


In [6]:
import sys
!{sys.executable} -m pip install llama-index-embeddings-huggingface

  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/784.9 kB ? eta -:--:--
   ---------------------------------------- 784.9/784.9 kB 6.7 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ------------------------------- -------- 3.1/4.0 MB 15.3 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 10.3 MB/s  0:00:00
   ---------------------------------------- 0.0/611.3 kB ? eta -:--:--
   ---------------------------------------- 611.3/611.3 kB 14.4 MB/s  0:00:00
   ----------------


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from llama_index.core import VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# 1. Download and set up a small, free local embedding model
print("Loading embedding model (this might take a minute to download)...")
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# 2. We don't have the main chat AI (LLM) connected yet, so we tell it to skip that for now
Settings.llm = None

# 3. Chop the documents into chunks and turn them into math (Vectors)!
print("Building the Vector Index...")
index = VectorStoreIndex.from_documents(documents)

print("Index built successfully! The searchable database is ready.")

C:\Users\agrim\PycharmProjects\pilotproject-deepfix\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model (this might take a minute to download)...


C:\Users\agrim\PycharmProjects\pilotproject-deepfix\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\agrim\AppData\Local\llama_index\llama_index\Cache\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8108.05it/s

LLM is explicitly disabled. Using MockLLM.
Building the Vector Index...
Index built successfully! The searchable database is ready.


In [9]:
# 1. Create a "retriever" from our index.
# similarity_top_k=1 means "give me the 1 best matching chunk of text"
retriever = index.as_retriever(similarity_top_k=1)

# 2. Let's test your exact theory!
query = "Who is the employee working on this?"
print(f"Asking the database: '{query}'\n")

# 3. Perform the search
retrieved_chunks = retriever.retrieve(query)

# 4. Print out what the database found, and how confident it is (the math score)
for chunk in retrieved_chunks:
    print(f"Retrieved Text (Similarity Score: {chunk.score:.3f}):")
    print(chunk.text)

Asking the database: 'Who is the employee working on this?'

Retrieved Text (Similarity Score: 0.510):
DeepFix is an autonomous AI framework designed to fix failing RAG systems.
It uses an AI agent named Hermes.
Agrim Sharma is the lead engineer responsible for building the retrieval pipeline.
DeepFix is a joint project between the AISC at HPI and Mittelstand Zukunftslab.


In [10]:
# 1. Upgrade our simple Retriever into a full Query Engine (Retriever + LLM)
print("Building Query Engine...\n")
query_engine = index.as_query_engine()

# 2. Ask the question
question = "Who is the employee working on this?"
print(f"User Question: {question}")

# 3. Generate the final answer!
response = query_engine.query(question)

# 4. Print the final AI response
print("\n=== Final AI Response ===")
print(response)

Building Query Engine...

User Question: Who is the employee working on this?

=== Final AI Response ===
Context information is below.
---------------------
file_path: C:\Users\agrim\PycharmProjects\pilotproject-deepfix\notebooks\..\data\raw\deepfix_facts_agrim.txt

DeepFix is an autonomous AI framework designed to fix failing RAG systems.
It uses an AI agent named Hermes.
Agrim Sharma is the lead engineer responsible for building the retrieval pipeline.
DeepFix is a joint project between the AISC at HPI and Mittelstand Zukunftslab.
---------------------
Given the context information and not prior knowledge, answer the query.
Query: Who is the employee working on this?
Answer: 
